In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import requests

# MongoDB를 파이썬과 연동하기

## MongoDB를 사용하기 위해서 pymongo 라이브러리를 설치하고 import 한다.

`!pip install pymongo`

In [2]:
import pymongo

## MongoDB에 연결하고 데이터베이스 목록을 얻어온다.

MongoClient 클래스로 MongoDB가 설치된 url을 넘겨서 MongoDB와 연결한다.

In [3]:
conn = pymongo.MongoClient('mongodb://localhost:27017')
conn

MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True)

MongoDB와 연결된 객체에서 list_database_names() 메소드를 실행하면 MongoDB의 데이터베이스 목록을 리스트로 얻어올 수 있다.

In [4]:
databaseList = conn.list_database_names()
print(type(databaseList))
databaseList

<class 'list'>


['admin', 'config', 'local', 'starbucks']

In [5]:
if 'starbucks' in databaseList:
    print('starbucks 데이터베이스가 있습니다.')
else:
    print('starbucks 데이터베이스가 없습니다.')

starbucks 데이터베이스가 있습니다.


## MongoDB의 데이터베이스를 사용할 준비를 한다.

starbucks라는 데이터베이스가 없으면 데이터베이스를 만들고 사용할 준비를 하고 있으면 사용할 준비를 한다.

In [6]:
starbucks = conn['starbucks'] # test> use starbucks
starbucks

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'starbucks')

sido라는 컬렉션이 없으면 컬렉션을 만들고 사용하고 있으면 그냥 사용한다.

In [7]:
sido = starbucks['sido'] # starbucks> db.createCollention('sido')
sido

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'starbucks'), 'sido')

sido 컬렉션에 저장된 모든 도큐먼트를 제거한다. delete_many() 메소드는 삭제 조건이 없더라도 반드시 `{}`를 입력해야 한다.

In [8]:
sido.delete_many({}) # starbucks> db.sido.deleteMany({})

DeleteResult({'n': 16, 'ok': 1.0}, acknowledged=True)

## 스타벅스에서 시도 정보를 크롤링하고 MongoDB에 저장하기 위해 딕셔너리 형태로 만든다.

스타벅스에서 시도 정보를 크롤링한다.

In [9]:
targetSite = 'https://www.starbucks.co.kr/store/getSidoList.do'
response = requests.post(targetSite).json()
sidoList = []
for res in response['list']:
    # print(res)
    dic = {
        'sido_cd': res['sido_cd'],
        'sido_nm': res['sido_nm']
    }
    # print(dic)
    sidoList.append(dic)
sidoList

[{'sido_cd': '01', 'sido_nm': '서울'},
 {'sido_cd': '08', 'sido_nm': '경기'},
 {'sido_cd': '03', 'sido_nm': '대구'},
 {'sido_cd': '04', 'sido_nm': '대전'},
 {'sido_cd': '05', 'sido_nm': '부산'},
 {'sido_cd': '06', 'sido_nm': '울산'},
 {'sido_cd': '07', 'sido_nm': '인천'},
 {'sido_cd': '09', 'sido_nm': '강원'},
 {'sido_cd': '10', 'sido_nm': '경남'},
 {'sido_cd': '11', 'sido_nm': '경북'},
 {'sido_cd': '12', 'sido_nm': '전남광주'},
 {'sido_cd': '13', 'sido_nm': '전북'},
 {'sido_cd': '14', 'sido_nm': '충남'},
 {'sido_cd': '15', 'sido_nm': '충북'},
 {'sido_cd': '16', 'sido_nm': '제주'},
 {'sido_cd': '17', 'sido_nm': '세종'}]

스타벅스 사이트에서 크롤링한 시도 정보를 MongoDB에 저장한다.

MongoDB의 starbucks 데이터베이스의 sido 컬렉션에 sidoList라는 리스트에 저장된 딕셔너리를 도큐먼트로 저장한다.

In [10]:
for doc in sidoList:
    # print(doc)
    sido.insert_one(doc) # starbucks> db.sido.insertOne({'sido_cd': '01', 'sido_nm': '서울'})

## MongoDB에 저장된 데이터를 읽어온다.

MongoDB와 연동하고 starbucks 데이터베이스를 사용 가능한 상태로 만든다.

In [11]:
conn = pymongo.MongoClient('mongodb://localhost:27017')
starbucks = conn['starbucks']
starbucks

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'starbucks')

starbucks 데이터베이스의 sido 컬렉션을 사용 가능한 상태로 만든다.

In [12]:
sido = starbucks['sido']
sido

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'starbucks'), 'sido')

sido 컬렉션에 저장된 모든 시도 코드 정보를 얻어온다.

In [13]:
sidoList = sido.find({}, {'_id': 0}) # starbucks> db.sido.find({}, {'_id': 0})
print(sidoList)

MongoBD에서 find() 메소드를 실행해서 데이터를 얻어오면 얻어온 데이터가 cursor에 저장되서 넘어온다.

cursor를 for로 반복시키면 cursor에 저장된 데이터를 읽어오는 next() 메소드를 실행할 필요없이 자동으로 실행된다.  
아래의 반복문을 실행하면 sidoList라는 cursor에 저장된 데이터가 소모될때마다 cursor가 1씩 증가해서 처리할 데이터를 얻어온다.

```
for doc in sidoList:  
    # print(type(doc))  
    print(doc)  
```

In [14]:
while True:
    try:
        print(sidoList.next())
    except:
        break

{'sido_cd': '01', 'sido_nm': '서울'}
{'sido_cd': '08', 'sido_nm': '경기'}
{'sido_cd': '03', 'sido_nm': '대구'}
{'sido_cd': '04', 'sido_nm': '대전'}
{'sido_cd': '05', 'sido_nm': '부산'}
{'sido_cd': '06', 'sido_nm': '울산'}
{'sido_cd': '07', 'sido_nm': '인천'}
{'sido_cd': '09', 'sido_nm': '강원'}
{'sido_cd': '10', 'sido_nm': '경남'}
{'sido_cd': '11', 'sido_nm': '경북'}
{'sido_cd': '12', 'sido_nm': '전남광주'}
{'sido_cd': '13', 'sido_nm': '전북'}
{'sido_cd': '14', 'sido_nm': '충남'}
{'sido_cd': '15', 'sido_nm': '충북'}
{'sido_cd': '16', 'sido_nm': '제주'}
{'sido_cd': '17', 'sido_nm': '세종'}


## MongoDB에 저장된 데이터를 수정한다.

```
starbucks> db.sido.updateOne(
    {'sido_cd': '11'},
    {
        $set: {'sido_nm': '경북대구통합특별시'}
    }
)
```

파이썬과 MongoDB의 문법 차이때문에 `$set` 부분에서 SyntaxError가 발생되므로 `$set` 부분을 따옴표로 감싸준다.

In [15]:
sido.update_one(
    {'sido_cd': '11'},
    {
        '$set': {'sido_nm': '경북대구통합특별시'}
    }
)

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

## MongoDB에 저장된 데이터를 제거한다.

In [16]:
sido.delete_one({'sido_cd': '03'}) # starbucks> db.sido.deleteOne({'sido_cd': '03'})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)